# Temperatura Colombia — Actualización Automática
Descarga datos reales desde **Open-Meteo API** (gratuita, sin API key).
Ejecuta la celda y el gráfico se actualiza con datos del día.

In [ ]:
# pip install requests pandas matplotlib
import requests
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from datetime import date
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# ── Configuración de ciudades ──────────────────────────────────────────────
CIUDADES = {
    'Barranquilla':  dict(lat=10.96, lon=-74.80, color='#FF6B35'),
    'Bogotá':        dict(lat= 4.71, lon=-74.07, color='#4E8098'),
    'Buenaventura':  dict(lat= 3.88, lon=-77.02, color='#2E8B57'),
    'Villavicencio': dict(lat= 4.14, lon=-73.63, color='#DAA520'),
    'Leticia':       dict(lat=-4.21, lon=-69.94, color='#CD853F'),
    'Medellín':      dict(lat= 6.25, lon=-75.57, color='#9370DB'),
    'Cali':          dict(lat= 3.44, lon=-76.52, color='#20B2AA'),
}

HOY        = date.today()
AÑO_ACTUAL = HOY.year
INICIO_HIST = f'{AÑO_ACTUAL - 5}-01-01'   # 5 años atrás
FIN_HIST    = f'{AÑO_ACTUAL - 1}-12-31'
INICIO_HOY  = f'{AÑO_ACTUAL}-01-01'
FIN_HOY     = str(HOY)

print(f'Descargando datos hasta: {FIN_HOY}')

In [ ]:
def fetch_temp(lat, lon, start, end):
    """Descarga temperatura media diaria desde Open-Meteo."""
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = dict(
        latitude=lat, longitude=lon,
        start_date=start, end_date=end,
        daily='temperature_2m_mean',
        timezone='America/Bogota'
    )
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    data = r.json()['daily']
    return pd.Series(
        data['temperature_2m_mean'],
        index=pd.to_datetime(data['time']),
        dtype=float
    )

hist_data = {}   # ciudad → Series diaria histórica
curr_data = {}   # ciudad → Series diaria año actual

for ciudad, cfg in CIUDADES.items():
    print(f'  {ciudad}...', end=' ', flush=True)
    hist_data[ciudad] = fetch_temp(cfg['lat'], cfg['lon'], INICIO_HIST, FIN_HIST)
    curr_data[ciudad] = fetch_temp(cfg['lat'], cfg['lon'], INICIO_HOY,  FIN_HOY)
    print('OK')

print('\nDescarga completada.')

In [ ]:
def monthly_stats(series):
    """Devuelve DataFrame con promedio, máx y mín por mes (1-12)."""
    m = series.groupby(series.index.month)
    return pd.DataFrame({'mean': m.mean(), 'max': m.max(), 'min': m.min()})

MESES = ['Ene','Feb','Mar','Abr','May','Jun','Jul','Ago','Sep','Oct','Nov','Dic']

fig = plt.figure(figsize=(18, 16))
fig.patch.set_facecolor('#0D1117')
gs = GridSpec(4, 2, figure=fig, hspace=0.55, wspace=0.35)

ciudades = list(CIUDADES.keys())
pos = [(0,0),(0,1),(1,0),(1,1),(2,0),(2,1),(3,0)]

for idx, ciudad in enumerate(ciudades):
    row, col = pos[idx]
    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor('#161B22')
    color = CIUDADES[ciudad]['color']

    # Estadísticas históricas por mes
    st = monthly_stats(hist_data[ciudad])
    x = np.arange(1, 13)

    ax.fill_between(x, st['min'], st['max'], alpha=0.25, color=color,
                    label=f'Rango {AÑO_ACTUAL-5}–{AÑO_ACTUAL-1}')
    ax.plot(x, st['mean'], '--', color=color, linewidth=1.5, alpha=0.8, label='Promedio hist.')

    # Serie año actual (promedio mensual hasta hoy)
    curr_monthly = curr_data[ciudad].groupby(curr_data[ciudad].index.month).mean()
    ax.plot(curr_monthly.index, curr_monthly.values, '-o', color='white',
            linewidth=2.2, markersize=6, label=f'{AÑO_ACTUAL} (actual)', zorder=5)

    # Último valor con anotación
    last_month = curr_monthly.index[-1]
    last_val   = curr_monthly.values[-1]
    ax.annotate(f'{last_val:.1f}°C', xy=(last_month, last_val),
                xytext=(5, 6), textcoords='offset points',
                color='white', fontsize=8, fontweight='bold')

    ax.set_title(ciudad, fontsize=11, fontweight='bold', color=color, pad=6)
    ax.set_xticks(x)
    ax.set_xticklabels(MESES, color='white', fontsize=7.5)
    ax.tick_params(colors='white')
    ax.spines[:].set_color('#30363D')
    ax.grid(True, alpha=0.15, color='gray', linestyle='--')
    ax.set_ylabel('°C', color='white', fontsize=9)
    if idx == 0:
        ax.legend(loc='lower right', framealpha=0.3, facecolor='#30363D',
                  labelcolor='white', fontsize=7.5)

# Panel resumen
ax_s = fig.add_subplot(gs[3, 1])
ax_s.set_facecolor('#161B22')
ax_s.axis('off')
lines = [f'Resumen  (datos al {FIN_HOY})', '']
for ciudad in ciudades:
    h = monthly_stats(hist_data[ciudad])
    c_avg = curr_data[ciudad].mean()
    lines.append(f'{ciudad:<15} mín {h["min"].min():.1f}°C  '
                 f'máx {h["max"].max():.1f}°C  '
                 f'{AÑO_ACTUAL} avg {c_avg:.1f}°C')
ax_s.text(0.05, 0.95, '\n'.join(lines), transform=ax_s.transAxes,
          va='top', fontsize=7.8, color='white', fontfamily='monospace',
          bbox=dict(boxstyle='round', facecolor='#21262D', alpha=0.8))

fig.suptitle(
    f'Colombia — Temperatura: Serie {AÑO_ACTUAL} vs Rango Histórico {AÑO_ACTUAL-5}–{AÑO_ACTUAL-1}',
    fontsize=14, fontweight='bold', color='white', y=0.98
)
fig.text(0.5, 0.005,
    f'Fuente: Open-Meteo Archive API · Actualizado: {FIN_HOY} · '
    f'Banda = rango máx/mín histórico · Línea blanca = {AÑO_ACTUAL}',
    ha='center', color='gray', fontsize=8)

plt.savefig('temperatura_colombia_auto.png', dpi=150, bbox_inches='tight', facecolor='#0D1117')
plt.show()
print(f'\nGráfico guardado — datos al {FIN_HOY}')